In [ ]:
# === 1. 라이브러리 임포트 및 Google 드라이브 연동 ===
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import time
import random
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive가 마운트되었습니다.")
    # 모델 저장 경로 설정
    MODEL_PATH = '/content/drive/My Drive/simple_vgg_trained.pth'
except ImportError:
    print("Google Colab 환경이 아닙니다. 로컬 경로에 모델을 저장합니다.")
    MODEL_PATH = './simple_vgg_trained.pth'

Mounted at /content/drive
Google Drive가 마운트되었습니다.


In [ ]:
# === 2. 기본 설정 (디바이스, 클래스) ===
# Colab의 GPU(CUDA) 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

Using device: cuda


In [ ]:
# === 3. SimpleVGG 모델 정의 (CIFAR-10 32x32용) ===
class SimpleVGG(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleVGG, self).__init__()
        # 32x32 입력
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),   # 3x3
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  # 3x3
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),      # 16x16

            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 3x3
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),# 3x3
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),      # 8x8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),# 3x3
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),# 3x3
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),      # 4x4
        )
        # 4x4x256 = 4096
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

print("SimpleVGG 모델 정의 완료.")

SimpleVGG 모델 정의 완료.


In [ ]:
# === 4. 데이터셋 및 DataLoader (CIFAR-10 네이티브 32x32) ===
BATCH_SIZE = 128

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
# 추론용 테스트 로더
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("CIFAR-10 데이터셋 준비 완료.")

100%|██████████| 170M/170M [00:04<00:00, 42.0MB/s]


CIFAR-10 데이터셋 준비 완료.


In [ ]:
# === 5. 모델 학습 (Training) ===
# (학습된 모델이 이미 있다면 이 섹션을 주석 처리하고 실행해도 됩니다)

model_for_train = SimpleVGG(num_classes=10).to(device)
NUM_EPOCHS = 10
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_for_train.parameters(), lr=LEARNING_RATE)

print(f"\nSimpleVGG 모델 학습을 시작합니다... (총 {NUM_EPOCHS} 에포크)")
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model_for_train.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_for_train(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (i + 1) % 100 == 0:
            print(f'Epoch [{epoch + 1}/{NUM_EPOCHS}], Step [{i + 1}/{len(train_loader)}], Loss: {running_loss / 100:.4f}')
            running_loss = 0.0

end_time = time.time()
print(f"\n학습 완료! (총 학습 시간: {end_time - start_time:.2f}초)")


SimpleVGG 모델 학습을 시작합니다... (총 10 에포크)
Epoch [1/10], Step [100/391], Loss: 2.1008
Epoch [1/10], Step [200/391], Loss: 1.8076
Epoch [1/10], Step [300/391], Loss: 1.6161
Epoch [2/10], Step [100/391], Loss: 1.3704
Epoch [2/10], Step [200/391], Loss: 1.2977
Epoch [2/10], Step [300/391], Loss: 1.2212
Epoch [3/10], Step [100/391], Loss: 1.0850
Epoch [3/10], Step [200/391], Loss: 1.0493
Epoch [3/10], Step [300/391], Loss: 1.0086
Epoch [4/10], Step [100/391], Loss: 0.9389
Epoch [4/10], Step [200/391], Loss: 0.9048
Epoch [4/10], Step [300/391], Loss: 0.8675
Epoch [5/10], Step [100/391], Loss: 0.8306
Epoch [5/10], Step [200/391], Loss: 0.7921
Epoch [5/10], Step [300/391], Loss: 0.8096
Epoch [6/10], Step [100/391], Loss: 0.7622
Epoch [6/10], Step [200/391], Loss: 0.7412
Epoch [6/10], Step [300/391], Loss: 0.7299
Epoch [7/10], Step [100/391], Loss: 0.6659
Epoch [7/10], Step [200/391], Loss: 0.6822
Epoch [7/10], Step [300/391], Loss: 0.7057
Epoch [8/10], Step [100/391], Loss: 0.6381
Epoch [8/10], St

In [ ]:
# === 6. 모델 저장 ===
try:
    torch.save(model_for_train.state_dict(), MODEL_PATH)
    print(f"학습된 SimpleVGG 모델이 '{MODEL_PATH}'에 저장되었습니다.")
except Exception as e:
    print(f"모델 저장 중 오류 발생: {e}")


print("\n" + "=" * 50)
print("     [ 양자화 검증 추론 단계 시작 ]")
print("=" * 50)


학습된 SimpleVGG 모델이 '/content/drive/My Drive/simple_vgg_trained.pth'에 저장되었습니다.

     [ 양자화 검증 추론 단계 시작 ]


In [ ]:
# === 7. 양자화 헬퍼 함수 ===

def quantize_symmetric_per_tensor(tensor, n_bits=8):
    """(시뮬레이션) 텐서를 대칭 INT8로 양자화합니다."""
    # FP32 텐서의 절대값 최대치를 찾아 스케일(scale) 계산
    # 127 = 2^(n_bits-1) - 1
    scale = torch.max(torch.abs(tensor.detach())) / (2**(n_bits - 1) - 1)

    # 스케일로 나누고 반올림하여 정수 텐서 생성
    quantized_tensor = torch.round(tensor / scale).clamp(-128, 127)

    return quantized_tensor, scale

def dequantize_symmetric_per_tensor(quantized_tensor, scale):
    """(시뮬레이션) 양자화된 텐서를 FP32로 복원합니다."""
    return quantized_tensor.float() * scale

In [ ]:
# === 8. 커스텀 Conv 포워딩 함수 (Debug Print 추가) ===

def custom_conv_forward(conv_layer, input_tensor, layer_index, print_debug=False):
    """
    모든 Conv 레이어에 대해 INT8 '정수 연산'을 시뮬레이션하고,
    print_debug=True일 때 텐서 값을 출력합니다.
    """
    weight_data = conv_layer.weight
    bias_data = conv_layer.bias

    # 1. 텐서 양자화 (INT8)
    input_q, scale_in = quantize_symmetric_per_tensor(input_tensor, n_bits=8)
    weight_q, scale_w = quantize_symmetric_per_tensor(weight_data, n_bits=8)

    # 2. Bias 양자화 (INT32)
    scale_bias = scale_in * scale_w
    bias_q = torch.round(bias_data / scale_bias)

    # --- [요청하신 디버그 PRINT문] ---
    if print_debug:
        print(f"\n--- [Debug] Conv Layer #{layer_index} ---")
        print(f"  Scales: Input_S={scale_in.item():.6f}, Weight_S={scale_w.item():.6f}, Bias_S(In*W)={scale_bias.item():.6f}")

        # Input (FP32 vs INT8) - 텐서의 맨 앞 3개 값 샘플
        print(f"  Input (FP32): {input_tensor.flatten()[0:3].cpu().numpy()}")
        print(f"  Input (INT8): {input_q.flatten()[0:3].cpu().numpy()}")

        # Weight (FP32 vs INT8) - 텐서의 맨 앞 3개 값 샘플
        print(f"  Weight (FP32): {weight_data.flatten()[0:3].cpu().numpy()}")
        print(f"  Weight (INT8): {weight_q.flatten()[0:3].cpu().numpy()}")

        # Bias (FP32 vs INT32) - 텐서의 맨 앞 3개 값 샘플
        print(f"  Bias (FP32): {bias_data[:3].cpu().numpy()}")
        print(f"  Bias (INT32): {bias_q[:3].cpu().numpy()}")
        print("  " + "-"*20)
    # --- [디버그 PRINT 종료] ---

    # 3. 정수 컨볼루션 시뮬레이션 -> 여기서는 sw로 하지만 나중에는 fpga로 보내고 결과 받아올 것!
    input_q_float = input_q.float()
    weight_q_float = weight_q.float()

    output_acc_sim_float = F.conv2d(input_q_float,
                                    weight_q_float,
                                    bias=None,
                                    stride=conv_layer.stride,
                                    padding=conv_layer.padding)

    # 4. 양자화된 Bias 더하기 (INT32 + INT32) -> 여기서는 sw로 하지만 나중에는 fpga로 보내고 결과 받아올 것!
    bias_q_reshaped = bias_q.reshape(1, -1, 1, 1)
    output_acc_with_bias_sim_float = output_acc_sim_float + bias_q_reshaped


    # 5. 최종 결과 역양자화 (INT32 -> FP32)
    output_tensor = output_acc_with_bias_sim_float * scale_bias

    return output_tensor

In [ ]:
# === 9. 텐서 검사를 포함하는 커스텀 추론 함수 ===
def custom_inference_with_quant(model, image_batch, print_debug=False):
    """
    모델의 'features' 모듈을 수동으로 실행하며 Conv2d를 가로채는 함수.
    """
    x = image_batch
    conv_layer_index = 0 # Conv 레이어 카운터

    for layer in model.features:
        if isinstance(layer, nn.Conv2d):
            # Conv2d 레이어는 '커스텀 포워딩' 함수 호출
            x = custom_conv_forward(layer, x, conv_layer_index, print_debug)
            conv_layer_index += 1 # 인덱스 증가
        else:
            # ReLU, MaxPool 등은 PyTorch로 실행
            x = layer(x)

    # Classifier는 PyTorch로 실행
    x = torch.flatten(x, 1) # [B, 256, 4, 4] -> [B, 4096]
    x = model.classifier(x)

    return x

In [ ]:
# === 10. 모델 로드 ===

print("\n=== 검증 프로세스 시작 ===")
inference_model = SimpleVGG(num_classes=10).to(device)

try:
    inference_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f"성공: '{MODEL_PATH}'에서 학습된 가중치를 불러왔습니다.")
except Exception as e:
    print(f"가중치 로드 실패: {e}")

inference_model.eval() # 추론 모드로 설정


=== 검증 프로세스 시작 ===
성공: '/content/drive/My Drive/simple_vgg_trained.pth'에서 학습된 가중치를 불러왔습니다.


SimpleVGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=4096, out_features=102

In [ ]:
# === 11. 전체 테스트셋 추론 및 정확도 비교 ===
# *** 수정: 첫 번째 배치(i=0)에만 print_debug=True로 설정 ***
print(f"\n테스트셋 전체 ({len(test_dataset)}개 이미지)에 대한 정확도 비교 시작...")

correct_a = 0 # [A] PyTorch 기본 추론 카운터
correct_b = 0 # [B] "모든" 레이어 양자화 추론 카운터
total = 0

with torch.no_grad():
    for i, (images, labe ls) in enumerate(test_loader):
        images, labels = images.to(device), labels.to(device)

        # --- [Debug Print Flag] ---
        # 첫 번째 배치(i=0)에 대해서만 디버그 프린트 활성화
        do_debug_print = (i == 0)
        if do_debug_print:
            print("\n" + "="*30)
            print(f"  [배치 0번] 디버그 프린트 활성화")
            print("="*30)
        # --- [Flag 종료] ---

        # (A) 기준점: 순수 PyTorch로만 추론
        output_a = inference_model(images)
        _, pred_a = torch.max(output_a.data, 1)

        # (B) 검증 대상: 모든 컨볼루션 레이어 양자화 (print_debug 플래그 전달)
        output_b = custom_inference_with_quant(inference_model, images, print_debug=do_debug_print)
        _, pred_b = torch.max(output_b.data, 1)

        # 카운트 업데이트
        total += labels.size(0)
        correct_a += (pred_a == labels).sum().item()
        correct_b += (pred_b == labels).sum().item()

        if (i + 1) % 20 == 0:
            print(f"  ... 배치 [{i+1}/{len(test_loader)}] 처리 중 ...")

# 최종 정확도 계산
accuracy_a = 100 * correct_a / total
accuracy_b = 100 * correct_b / total

print("\n" + "=" * 50)
print("       최종 정확도 비교 결과 (SimpleVGG)")
print("=" * 50)
print(f"  - 총 테스트 이미지: {total}개")
print(f"  - [A] PyTorch FP32 추론 정확도: \t{accuracy_a:.2f} % ({correct_a}/{total})")
print(f"  - [B] *모든* Conv 레이어 INT8 양자화 추론 정확도: \t{accuracy_b:.2f} % ({correct_b}/{total})")
print("-" * 50)
print(f"  - 정확도 차이 (A - B): {accuracy_a - accuracy_b:+.2f} %")

print("\nColab 통합 스크립트 종료.")




테스트셋 전체 (10000개 이미지)에 대한 정확도 비교 시작...

  [배치 0번] 디버그 프린트 활성화

--- [Debug] Conv Layer #0 ---
  Scales: Input_S=0.021683, Weight_S=0.002273, Bias_S(Acc)=0.000049
  Input (FP32): [0.63375115 0.6531361  0.7694456 ]
  Input (INT8): [29. 30. 35.]
  Weight (FP32): [ 0.10153796 -0.07790072 -0.07311213]
  Weight (INT8): [ 45. -34. -32.]
  Bias (FP32): [0.00181552 0.04103457 0.14848216]
  Bias (INT8_temp): [ 1. 15. 55.]
  Bias (INT32_final): [  55.  827. 3031.]
  --------------------

--- [Debug] Conv Layer #1 ---
  Scales: Input_S=0.029354, Weight_S=0.002447, Bias_S(Acc)=0.000072
  Input (FP32): [0.         0.19637348 0.17148176]
  Input (INT8): [0. 7. 6.]
  Weight (FP32): [-0.11760495 -0.11479476 -0.04322489]
  Weight (INT8): [-48. -47. -18.]
  Bias (FP32): [ 0.06512188 -0.07216977 -0.01261251]
  Bias (INT8_temp): [ 74. -82. -14.]
  Bias (INT32_final): [  911. -1010.  -172.]
  --------------------

--- [Debug] Conv Layer #2 ---
  Scales: Input_S=0.075279, Weight_S=0.003425, Bias_S(Acc)=0.0002

# 새 섹션